# Step 3: stronger baselines and a mechanism test

Answers the three reviewer attacks left after Step 2. Datasets: DermaMNIST, BloodMNIST (fast). Run all cells; check `smoke: False`.

1. **Labeled-budget recalibration:** instead of recalibrating with all calibration labels, use only n = 50, 100 or 200 labeled examples. If a small label budget matches `sizematch`, the label-free method is not worth much. Both the mean gap and the **spread of coverage across splits** are reported, because small-n recalibration is noisy.
2. **RAPS-style score** added next to LAC and APS (fixed lambda = 0.05, k_reg = 2, no randomization, **not tuned**).
3. **Fine-tune dose-response:** unpruned fp32 fine-tuned for 1, 2 and 4 epochs, and pruned-50% fine-tuned 2 and 4 epochs. If undercoverage grows with epochs, that supports "extra training makes the model overconfident" as the mechanism (still correlational).

`sizematch` is label-free and has **no finite-sample guarantee**.

In [ ]:
!pip install -q medmnist scipy
import os, sys, copy, math, time, glob, itertools
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

# ---- CONFIG ----
SMOKE = False                      
DATASETS = ["dermamnist", "bloodmnist"]   
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = {"pathmnist": 8, "dermamnist": 15, "bloodmnist": 12}
FT_EPOCHS = 2                      
LABEL_BUDGETS = [50, 100, 200]     
N_SPLITS = 40                     
ALPHA = 0.1
USE_DRIVE = False                  
OUT_DIR = "/content/step3_out"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive"); OUT_DIR = "/content/drive/MyDrive/step3_out"
os.makedirs(OUT_DIR, exist_ok=True)
print("device:", DEVICE, "| smoke:", SMOKE, "| datasets:", DATASETS, "| seeds:", SEEDS, "| out:", OUT_DIR)
if DEVICE == "cpu": print("WARNING: CPU only. Switch to GPU (Runtime > Change runtime type) or run one dataset and one seed.")

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

## Code

In [ ]:
# %%
def lac_all(probs):
    """Score of every label: 1 - p(label). probs: (n, K)."""
    return 1.0 - probs

def aps_all(probs):
    """Deterministic APS score of every label: total prob mass of labels ranked at or above it."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    return np.take_along_axis(cum, ranks, axis=1)

def raps_all(probs, lam=0.05, k_reg=2):
    """Deterministic RAPS-style score: APS mass plus lam * max(0, rank - k_reg). Fixed hyperparameters, not tuned."""
    order = np.argsort(-probs, axis=1)
    cum = np.cumsum(np.take_along_axis(probs, order, axis=1), axis=1)
    ranks = np.argsort(order, axis=1)
    pen = lam * np.maximum(0, np.arange(1, probs.shape[1] + 1)[None, :] - k_reg)
    return np.take_along_axis(cum + pen, ranks, axis=1)

SCORES = {"LAC": lac_all, "APS": aps_all, "RAPS": raps_all}

def conformal_q(true_scores, alpha=ALPHA):
    """Finite-sample split-conformal quantile. Returns inf if the calibration set is too small."""
    n = len(true_scores)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    if n == 0 or k > n:
        return np.inf
    return float(np.sort(true_scores)[k - 1])

def summarize(sets, y, min_class_n=10):
    covered = sets[np.arange(len(y)), y]
    cls = [covered[y == c].mean() for c in np.unique(y) if (y == c).sum() >= min_class_n]
    return dict(coverage=float(covered.mean()), size=float(sets.sum(1).mean()),
                worst_class_cov=float(min(cls)) if cls else np.nan)


In [ ]:
# %%
class Block(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm2d(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.b2 = nn.BatchNorm2d(cout)
        self.sc = None
        if stride != 1 or cin != cout:
            self.sc = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        o = F.relu(self.b1(self.c1(x))); o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))

class ResNet18Small(nn.Module):
    """ResNet-18 layout (2,2,2,2 blocks) with a 3x3 stem, suited to 28x28 images."""
    def __init__(self, num_classes, width=64):
        super().__init__()
        w = width
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        cfg, layers, cin = [(w, 1), (w, 1), (2*w, 2), (2*w, 1), (4*w, 2), (4*w, 1), (8*w, 2), (8*w, 1)], [], w
        for cout, s in cfg:
            layers.append(Block(cin, cout, s)); cin = cout
        self.layers = nn.Sequential(*layers); self.fc = nn.Linear(cin, num_classes)
    def forward(self, x):
        x = self.layers(self.stem(x))
        return self.fc(F.adaptive_avg_pool2d(x, 1).flatten(1))

def train_model(model, X, y, epochs, lr=2e-3, bs=256, seed=0):
    set_seed(seed); model.to(DEVICE).train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            xb, yb = Xt[idx].to(DEVICE), yt[idx].to(DEVICE)
            loss = F.cross_entropy(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def predict_probs(model, X, bs=512):
    model.to(DEVICE).eval(); out = []
    Xt = torch.as_tensor(X)
    for i in range(0, len(Xt), bs):
        out.append(F.softmax(model(Xt[i:i + bs].to(DEVICE)), dim=1).cpu())
    return torch.cat(out).numpy().astype(np.float64)

WEIGHT_LAYERS = (nn.Conv2d, nn.Conv1d, nn.Linear)

def quantize_weights(model, bits):
    """Simulated symmetric per-output-channel weight quantization (weights only)."""
    m, qmax = copy.deepcopy(model), 2 ** (bits - 1) - 1
    for mod in m.modules():
        if isinstance(mod, WEIGHT_LAYERS):
            w = mod.weight.data; dims = tuple(range(1, w.dim()))
            scale = w.abs().amax(dim=dims, keepdim=True).clamp_min(1e-8) / qmax
            mod.weight.data = torch.round(w / scale).clamp(-qmax, qmax) * scale
    return m

def prune_global(model, sparsity):
    import torch.nn.utils.prune as prune
    m = copy.deepcopy(model)
    params = [(mod, "weight") for mod in m.modules() if isinstance(mod, WEIGHT_LAYERS)]
    prune.global_unstructured(params, pruning_method=prune.L1Unstructured, amount=sparsity)
    for mod, name in params: prune.remove(mod, name)
    return m

def finetune_masked(model, X, y, epochs, lr=5e-4, bs=256, seed=0):
    """Short fine-tune of a pruned model; zeroed weights stay zero (mask held fixed)."""
    set_seed(seed + 99); model.to(DEVICE).train()
    mods = [m for m in model.modules() if isinstance(m, WEIGHT_LAYERS)]
    masks = [(m.weight.data != 0).float() for m in mods]
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * math.ceil(len(y) / bs))
    Xt, yt = torch.as_tensor(X), torch.as_tensor(y).long()
    for ep in range(epochs):
        perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            idx = perm[i:i + bs]
            loss = F.cross_entropy(model(Xt[idx].to(DEVICE)), yt[idx].to(DEVICE))
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            with torch.no_grad():
                for m, mk in zip(mods, masks): m.weight.data *= mk
    return model.eval()

def make_comps(Xtr, ytr, seed):
    ftp = lambda s, e: (lambda m: finetune_masked(prune_global(m, s), Xtr, ytr, e, seed=seed))
    ftf = lambda e: (lambda m: finetune_masked(copy.deepcopy(m), Xtr, ytr, e, seed=seed))
    return {
        "fp32": lambda m: m,
        "fp32+ft1": ftf(1), "fp32+ft2": ftf(2), "fp32+ft4": ftf(4),
        "prune50": lambda m: prune_global(m, 0.5),
        "prune50+ft2": ftp(0.5, 2), "prune50+ft4": ftp(0.5, 4),
    }

def mean_size(S, q):
    return float((S <= q).sum(1).mean())

def size_match_q(S_c, target_size, iters=40):
    """Label-free: smallest threshold whose mean set size on unlabeled inputs reaches target_size (bisection)."""
    lo, hi = 0.0, float(S_c.max()) + 1e-9
    for _ in range(iters):
        mid = (lo + hi) / 2
        if mean_size(S_c, mid) < target_size: lo = mid
        else: hi = mid
    return hi

## Data and experiment

In [ ]:
def load_medmnist(name, smoke):
    if smoke:
        rng = np.random.RandomState(0); K = 7
        def make(n):
            y = rng.randint(0, K, n); X = rng.rand(n, 3, 28, 28).astype(np.float32) * 0.8
            for k in range(K):
                r, c, ch = (k // 3) * 9 + 2, (k % 3) * 9 + 2, k % 3
                X[y == k, ch, r:r + 8, c:c + 8] += 0.6
            return (X - 0.5) / 0.5, y
        Xtr, ytr = make(2500); Xev, yev = make(1500)
        return Xtr, ytr, Xev, yev, K
    import medmnist
    from medmnist import INFO
    cls = getattr(medmnist, INFO[name]["python_class"])
    f = lambda a: ((a.imgs.astype(np.float32) / 255.0 - 0.5) / 0.5).transpose(0, 3, 1, 2).copy()
    lab = lambda a: a.labels.squeeze().astype(np.int64)
    tr, va, te = (cls(split=s, download=True) for s in ("train", "val", "test"))
    K = len(INFO[name]["label"])
    if name == "pathmnist":      # test split is from a different center: use test only (random halves)
        Xev, yev = f(te), lab(te)
    else:                        # random MedMNIST splits: pool val+test for more calibration data
        Xev, yev = np.concatenate([f(va), f(te)]), np.concatenate([lab(va), lab(te)])
    return f(tr), lab(tr), Xev, yev, K

def run_dataset(name):
    Xtr, ytr, Xev, yev, K = load_medmnist(name, SMOKE)
    print(f"[{name}] train {len(ytr)} | calib+eval pool {len(yev)} | classes {K} | class counts (pool) {np.bincount(yev, minlength=K).tolist()}")
    epochs = 3 if SMOKE else EPOCHS[name]
    rows = []
    for seed in SEEDS:
        t0 = time.time()
        model = train_model(ResNet18Small(K, width=16 if SMOKE else 64), Xtr, ytr, epochs, seed=seed)
        comps = make_comps(Xtr, ytr, seed)
        probs = {c: predict_probs(fn(model), Xev) for c, fn in comps.items()}
        print(f"  seed {seed} ({time.time()-t0:.0f}s) acc: " + " ".join(f"{c}={np.mean(probs[c].argmax(1)==yev):.3f}" for c in comps), flush=True)
        for sname, sfn in SCORES.items():
            S = {c: sfn(p) for c, p in probs.items()}
            for sp in range(N_SPLITS):
                rng = np.random.RandomState(1000 * seed + sp)
                perm = rng.permutation(len(yev)); cal, ev = perm[:len(yev) // 2], perm[len(yev) // 2:]
                q_full = conformal_q(S["fp32"][cal, yev[cal]])
                target = mean_size(S["fp32"][cal], q_full)          # fp32 set size on UNLABELED calibration inputs
                for c in comps:
                    q_c = conformal_q(S[c][cal, yev[cal]])
                    q_sm = size_match_q(S[c][cal], target)             # uses no labels
                    qs = [("reuse", q_full), ("sizematch", q_sm), ("recal", q_c)]
                    for nb_ in LABEL_BUDGETS:
                        sub = np.random.RandomState(5000 * seed + sp + nb_).choice(cal, size=min(nb_, len(cal)), replace=False)
                        qs.append((f"recal_n{nb_}", conformal_q(S[c][sub, yev[sub]])))
                    for mode, q in qs:
                        r = summarize(S[c][ev] <= q, yev[ev], min_class_n=5)
                        rows.append(dict(dataset=name, seed=seed, split=sp, score=sname, comp=c, mode=mode,
                                         acc=float(np.mean(probs[c].argmax(1) == yev)), **r))
    df = pd.DataFrame(rows); df.to_csv(os.path.join(OUT_DIR, f"{name}_raw.csv"), index=False)
    return df

## Reporting

In [ ]:
from scipy import stats
COMP_ORDER = ["fp32", "fp32+ft1", "fp32+ft2", "fp32+ft4", "prune50", "prune50+ft2", "prune50+ft4"]
METHODS = ["reuse", "sizematch"] + [f"recal_n{n}" for n in LABEL_BUDGETS]
def ci95(x):
    x = np.asarray(x, float); n = len(x)
    return float("nan") if n < 2 else stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
def pm(x): return "{:+.2f} +/- {:.2f}".format(np.mean(x), ci95(x))

def report(df):
    if SMOKE: print("*** SMOKE = True: SYNTHETIC DATA, NOT A RESULT ***")
    name = df["dataset"].iloc[0]
    per = df.groupby(["seed", "score", "comp", "mode"])["coverage"].mean().reset_index()
    per["gap"] = 100 * (per["coverage"] - (1 - ALPHA))
    rows, rows_sd = [], []
    sd = df.groupby(["seed", "score", "comp", "mode"])["coverage"].std().reset_index()
    for (sc, c), g in per.groupby(["score", "comp"]):
        r = dict(score=sc, comp=c)
        for m in METHODS: r[m] = pm(g[g["mode"] == m]["gap"])
        r["_o"] = COMP_ORDER.index(c); rows.append(r)
        s = sd[(sd["score"] == sc) & (sd["comp"] == c)]
        r2 = dict(score=sc, comp=c, **{m: round(100 * s[s["mode"] == m]["coverage"].mean(), 2) for m in METHODS + ["recal"]}, _o=COMP_ORDER.index(c)); rows_sd.append(r2)
    o = pd.DataFrame(rows).sort_values(["score", "_o"]).drop(columns="_o")
    o2 = pd.DataFrame(rows_sd).sort_values(["score", "_o"]).drop(columns="_o")
    print(f"\n===== {name}: coverage gap (points vs 90), mean +/- 95% t-interval over {per['seed'].nunique()} seeds =====")
    print(o.to_string(index=False))
    print(f"\n----- {name}: spread of coverage ACROSS CALIBRATION SPLITS (std, points); lower = more reliable -----")
    print(o2.to_string(index=False))
    return per

def summarize_all(pers):
    allp = pd.concat([p.assign(dataset=n) for n, p in pers.items()])
    allp["abs_gap"] = allp["gap"].abs()
    a = allp[allp["mode"].isin(METHODS)].groupby(["score", "comp", "mode"])["gap"].apply(lambda x: x.abs().mean()).reset_index()
    t = a.pivot_table(index=["score", "comp"], columns="mode", values="gap")[METHODS].round(2)
    t = t.loc[[(s, c) for s in SCORES for c in COMP_ORDER if (s, c) in t.index]]
    print("\n===== SUMMARY: mean absolute coverage gap (points), averaged over seeds and datasets =====")
    print(t.to_string())
    d = allp[(allp["mode"] == "reuse") & (allp["comp"].str.startswith("fp32+ft") | (allp["comp"] == "fp32"))]
    dd = d.groupby(["score", "dataset", "comp"])["gap"].mean().reset_index().pivot_table(index=["score", "dataset"], columns="comp", values="gap")
    cols = [c for c in ["fp32", "fp32+ft1", "fp32+ft2", "fp32+ft4"] if c in dd.columns]
    print("\n===== DOSE-RESPONSE: reuse gap (points) of unpruned fp32 after 0/1/2/4 fine-tune epochs (negative = undercoverage) =====")
    print(dd[cols].round(2).to_string())
    print("\nRead: (a) does recal_n100/n200 beat sizematch on |gap| AND on spread? (b) does the gap grow with epochs?")

## Run

In [ ]:
outs, dfs = {}, []
for name in DATASETS:
    fpath = os.path.join(OUT_DIR, f"{name}_raw.csv")
    if os.path.exists(fpath) and not SMOKE:
        print(f"[{name}] found saved results, loading (delete {fpath} to rerun)"); df = pd.read_csv(fpath)
    else:
        df = run_dataset(name)
    dfs.append(df); outs[name] = report(df)
summarize_all(outs)
pd.concat(dfs).to_csv("step3_all_raw.csv", index=False)